In [ ]:
import numpy as np
import pandas as pd
import os
import cv2
from PIL import Image
from torchvision import transforms

In [ ]:
#路徑處理

train_dir = "xray_data/train"
test_dir  = "xray_data/test"

print("Train exists:", os.path.exists(train_dir))
print("Test exists:", os.path.exists(test_dir))

print("Train classes:", os.listdir(train_dir)[:15])
print("Test samples:", os.listdir(test_dir)[:10])

In [ ]:
# 影像預處理方法

IMG_SIZE = 224

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# =========================
# Image Processing Classes
# =========================

class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        img = np.array(img.convert("L"))

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )

        img = clahe.apply(img)
        img = Image.fromarray(img).convert("RGB")
        return img


class SharpenTransform:
    def __call__(self, img):
        img = np.array(img)

        kernel = np.array([
            [0, -1, 0],
            [-1, 5, -1],
            [0, -1, 0]
        ])

        sharpened = cv2.filter2D(img, -1, kernel)
        sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)

        return Image.fromarray(sharpened)


class GammaCorrection:
    def __init__(self, gamma=0.8):
        self.gamma = gamma

    def __call__(self, img):
        img = np.array(img).astype(np.float32) / 255.0
        img = np.power(img, self.gamma)
        img = np.clip(img * 255, 0, 255).astype(np.uint8)

        return Image.fromarray(img).convert("RGB")


class DenoiseTransform:
    def __call__(self, img):
        img = np.array(img.convert("L"))
        denoised = cv2.GaussianBlur(img, (3, 3), 0)

        return Image.fromarray(denoised).convert("RGB")

class HybridXrayEnhancement:
    def __init__(self, gamma=0.9, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.gamma = gamma
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        # 1. Convert to grayscale
        img = np.array(img.convert("L"))

        # 2. Light denoising
        img = cv2.GaussianBlur(img, (3, 3), 0)

        # 3. CLAHE local contrast enhancement
        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )
        img = clahe.apply(img)

        # 4. Gamma correction
        img = img.astype(np.float32) / 255.0
        img = np.power(img, self.gamma)
        img = np.clip(img * 255, 0, 255).astype(np.uint8)

        # 5. Light sharpening
        kernel = np.array([
            [0, -0.5, 0],
            [-0.5, 3.0, -0.5],
            [0, -0.5, 0]
        ])
        img = cv2.filter2D(img, -1, kernel)
        img = np.clip(img, 0, 255).astype(np.uint8)

        # 6. Convert back to RGB for pretrained models
        return Image.fromarray(img).convert("RGB")


# =========================
# Select One Experiment
# =========================

# 五個擇一，把要跑的那個取消註解

EXPERIMENT = "baseline"
# EXPERIMENT = "clahe"
# EXPERIMENT = "clahe_sharpen"
# EXPERIMENT = "gamma"
# EXPERIMENT = "denoise_clahe"
# EXPERIMENT = "hybrid"

# 是否加入資料增強
USE_AUGMENTATION = True
# USE_AUGMENTATION = False


# =========================
# Build preprocessing pipeline
# =========================

def get_preprocess_ops(experiment):

#  回傳主要影像處理方法
    
    if experiment == "baseline":
        return [
            transforms.Grayscale(num_output_channels=3)
        ]

    elif experiment == "clahe":
        return [
            CLAHETransform()
        ]

    elif experiment == "clahe_sharpen":
        return [
            CLAHETransform(),
            SharpenTransform()
        ]

    elif experiment == "gamma":
        return [
            transforms.Grayscale(num_output_channels=3),
            GammaCorrection(gamma=0.8)
        ]

    elif experiment == "denoise_clahe":
        return [
            DenoiseTransform(),
            CLAHETransform()
        ]

    elif experiment == "hybrid":
        return [
            HybridXrayEnhancement(
                gamma=0.9,
                clip_limit=2.0,
                tile_grid_size=(8, 8)
            )
        ]

    else:
        raise ValueError(f"Unknown experiment: {experiment}")


preprocess_ops = get_preprocess_ops(EXPERIMENT)


# Train transform
if USE_AUGMENTATION:
    train_transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=7),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        *preprocess_ops,
        transforms.ToTensor(),
        normalize
    ])
else:
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        *preprocess_ops,
        transforms.ToTensor(),
        normalize
    ])


# Validation / Test transform
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    *preprocess_ops,
    transforms.ToTensor(),
    normalize
])


print("Current experiment:", EXPERIMENT)
print("Use augmentation:", USE_AUGMENTATION)

In [ ]:
# dataset的建立用來訓練讀檔的
from torchvision import datasets
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np

base_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=None
)

targets = np.array(base_dataset.targets)
indices = np.arange(len(base_dataset))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=targets
)

train_full = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

val_full = datasets.ImageFolder(
    root=train_dir,
    transform=val_test_transform
)

train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(val_full, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Total images:", len(base_dataset))
print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))
print("Classes:", base_dataset.classes)